In [0]:
# tests/test_transformations.py
import pytest
from src.transformations import clean_patients, filter_invalid_claims

def test_clean_patients_name_and_state(spark):
    # 1. Create mock Bronze data
    data = [
        ("p1", "John", "Doe", "Pittsfield MA US"),
        ("p2", "Jane", "Smith", "New Bedford MA US"),
        (None, "Invalid", "User", "Boston MA US")  # Should be filtered out
    ]
    schema = ["patient", "first", "last", "birthplace"]
    df_raw = spark.createDataFrame(data, schema)

    # 2. Execute transformation
    df_result = clean_patients(df_raw)
    rows = df_result.collect()

    # 3. Assert expected output
    assert df_result.count() == 2
    assert rows[0]["patient_name"] == "John Doe"
    assert rows[0]["state"] == "MA"
    assert rows[1]["patient_name"] == "Jane Smith"


def test_filter_invalid_claims(spark):
    # 1. Mock claim data
    data = [
        ("c1", 100.0),
        ("c2", -50.0),   # Invalid: negative cost
        (None, 200.0)    # Invalid: missing ID
    ]
    schema = ["claim_id", "total_cost"]
    df_raw = spark.createDataFrame(data, schema)

    # 2. Execute transformation
    df_result = filter_invalid_claims(df_raw)

    # 3. Assertions
    assert df_result.count() == 1
    assert df_result.first()["claim_id"] == "c1"